# Download IG price history (IG demo account)

Pulls **OHLC candles** for an IG market (`EPIC`, `RESOLUTION`, `DAYS_BACK` —
default is `IX.D.NASDAQ.IFA.IP` / `DAY` / the last 1 day) over a recent window
from the IG REST API, then can save them to a timestamped
`data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv` and/or upsert them into the
resolution-specific `candles_<suffix>` table (e.g. `candles_1d`) of
`data/ig_market_data.db` (same schema the sibling `03_view_ig_prices.ipynb`
reads) — each output is independently toggled via `SAVE_CSV` / `SAVE_DB` in
the config cell.

**Credentials and config** are read from a `.env` file in this folder
(git-ignored). Keys expected:

| var | meaning |
| --- | --- |
| `IG_API_KEY` | IG API key for the **demo** environment |
| `IG_USERNAME` | IG demo login |
| `IG_PASSWORD` | IG demo password |
| `IG_ACCOUNT_TYPE` | `demo` or `live` (default `demo`) |
| `IG_EPIC` | market to pull candles for (default `IX.D.NASDAQ.IFA.IP`) |
| `IG_RESOLUTION` | IG candle resolution (default `DAY`) |
| `IG_DAYS_BACK` | lookback window in calendar days (default `1`) |
| `IG_SAVE_CSV` | write the timestamped CSV (`true`/`false`, default `true`) |
| `IG_SAVE_DB` | upsert into the SQLite DB (`true`/`false`, default `false`) |

The download needs `requests` + `python-dotenv`. `pandas` / `matplotlib` are
used for the optional preview and chart at the end.

## 1. Load config from `.env`

Uses [`python-dotenv`](https://pypi.org/project/python-dotenv/) - run the
install line once if it's not already available. `Config.from_env()` (in
`config.py`) loads `.env`, reads the IG credentials and notebook settings
listed above (with defaults where noted), and returns them as a single `cfg`
object used by every cell below.

In [1]:
# One-time setup: install this repo's `igmarket` package (editable) + deps.
# Walks up to the repo root so it works whether the kernel's working
# directory is the repo root or the notebooks/ folder. Re-running is cheap.
import subprocess, sys
from pathlib import Path

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{_root}[dev]"],
    check=True,
)

CompletedProcess(args=['C:\\Users\\justi\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pip', 'install', '-q', '-e', 'C:\\repos\\github.com\\jupyter-notebooks[dev]'], returncode=0)

In [2]:
import sqlite3
from datetime import datetime, timedelta, timezone
from pathlib import Path

from igmarket.config import Config

cfg = Config.from_env()

print(f"Loaded config from {cfg.env_path.resolve()}")
print(f"user={cfg.username!r}  account_type={cfg.account_type!r}  api_key=***{cfg.api_key[-4:]}")
print(f"EPIC={cfg.epic!r}  RESOLUTION={cfg.resolution!r}  DAYS_BACK={cfg.days_back}  SAVE_CSV={cfg.save_csv}  SAVE_DB={cfg.save_db}")

Loaded config from C:\repos\github.com\jupyter-notebooks\.env
user='justinjingsh-demo'  account_type='demo'  api_key=***9251
EPIC='IX.D.NASDAQ.IFA.IP'  RESOLUTION='DAY'  DAYS_BACK=1  SAVE_CSV=False  SAVE_DB=False


## 2. Download

Opens an authenticated `IGSession` (`ig_session.py` — `POST /session` to
authenticate, then `GET /prices/{epic}` (`Version: 3`) with
`resolution`/`from`/`to`/`pageSize`/`pageNumber` to page through history;
mirrors the `IGSession` class in
`C:\repos\github.com\ig-quant-trading\src\ig_quant_trading\client.py`), then
fetches `RESOLUTION` candles for `EPIC` over the trailing `DAYS_BACK` day(s)
via `get_prices()`. Prints the candle count, the UTC time range covered, and
the historical-data allowance remaining on the account — the raw
`metadata.allowance` field names (`remainingAllowance`, `totalAllowance`,
`allowanceExpiry`) live in `constants/allowance_fields.py` (`AllowanceField`).

In [3]:
from igmarket.constants.allowance_fields import AllowanceField
from igmarket.constants.ig_candle_fields import CandleField
from igmarket.ig_session import IGSession

ig = IGSession(cfg.api_key, cfg.username, cfg.password, cfg.account_type)

fmt   = "%Y-%m-%dT%H:%M:%S"
end   = datetime.now(timezone.utc)
start = end - timedelta(days=cfg.days_back)

raw_candles, allowance = ig.get_prices(
    cfg.epic, cfg.resolution, start.strftime(fmt), end.strftime(fmt)
)

print(f"{len(raw_candles)} {cfg.resolution} candles for {cfg.epic}")
if raw_candles:
    print(f"  range (UTC): {raw_candles[0][CandleField.SNAPSHOT_TIME_UTC]}  ->  {raw_candles[-1][CandleField.SNAPSHOT_TIME_UTC]}")
if allowance:
    print(
        f"  historical-data allowance: {allowance.get(AllowanceField.REMAINING_ALLOWANCE)}"
        f" / {allowance.get(AllowanceField.TOTAL_ALLOWANCE)} remaining"
        f" (resets in {allowance.get(AllowanceField.ALLOWANCE_EXPIRY)}s)"
    )

Authenticated OK
1 DAY candles for IX.D.NASDAQ.IFA.IP
  range (UTC): 2026-09-06T14:00:00  ->  2026-09-06T14:00:00
  historical-data allowance: 6872 / 10000 remaining (resets in 362915s)


## 3. Persist

Two outputs, each independently toggled in the config cell — `SAVE_CSV` and
`SAVE_DB` — so a run can write either, both, or neither.

### CSV — full bid/ask/mid OHLC

Toggled by `SAVE_CSV`. Row-flattening lives in `candle_csv.py`
(`candle_to_row()`); the raw IG field-name constants (`CandleField`,
`PriceField`) live in `constants/ig_candle_fields.py`, and column headers in
`constants/csv_headers.py` (`CSV_HEADERS`) — imported by the cells below so
they can't drift out of sync. Columns:

- `snapshot_time_utc` — IG's `snapshotTimeUTC`, ISO `YYYY-MM-DDTHH:MM:SS`, UTC
  (not the exchange-local `snapshotTime`)
- `{open,high,low,close}_{bid,ask,mid}_price` — 12 price columns, raw IG
  values; `mid` is derived (`(bid + ask) / 2`), not requested separately
- `last_traded_volume`

Each run writes a **new, timestamped file**
(`data/ig_<epic-slug>_<resolution-suffix>_<RUN_TIMESTAMP>.csv` — `_epic_slug()`
in `config.py` takes the instrument segment of the dot-separated `EPIC`, e.g.
`IX.D.NASDAQ.IFA.IP` -> `nasdaq`, and `RESOLUTION` maps to a filename-friendly
suffix via `constants/csv_filename_suffix.py`'s `RESOLUTION_CSV_SUFFIX`, e.g.
`DAY` -> `daily`, `MINUTE_10` -> `10min`) rather than overwriting a fixed
path, so nothing is ever lost between runs — but successive runs' CSVs
accumulate in `data/` and are never cleaned up automatically.

In [4]:
# --- CSV (full bid/ask/mid OHLC) ----------------------------------------------
def write_csv():
    if not cfg.save_csv:
        print("SAVE_CSV is False - skipping CSV write")
        return

    import csv

    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    with cfg.csv_path.open("w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(CSV_HEADERS)
        for c in raw_candles:
            writer.writerow(candle_to_row(c))
    print(f"wrote {cfg.csv_path.resolve()}  ({len(raw_candles)} rows)")


write_csv()

SAVE_CSV is False - skipping CSV write


### SQLite — upsert into `candles_<suffix>`

Toggled by `SAVE_DB`.

- **Target:** a resolution-specific table — `candles_1d` for `DAY`,
  `candles_10min` for `MINUTE_10`, etc. — in `data/ig_market_data.db`. Table
  creation and the `resolution` -> table-name mapping live in `candle_db.py`
  (`init_candles_table()`, `table_name_for_resolution()`), backed by
  `constants/resolutions.py` (`RESOLUTION_TABLE_SUFFIX`), so
  `03_view_ig_prices.ipynb` derives the same table name from a resolution.
- **Columns:** `epic`, then the flattened price columns matching the CSV
  output today — `constants/db_headers.py`'s `DB_HEADERS`, a copy of
  `constants/csv_headers.py`'s `CSV_HEADERS` kept as its own list so the two
  schemas can diverge independently — built with the same
  `candle_csv.candle_to_row()` used for the CSV. The resolution isn't stored
  — it's fixed per table and recoverable from the table name.
- **Key:** `snapshot_time_utc` (UTC, from IG's `snapshotTimeUTC` — not the
  exchange-local `snapshotTime`); `UNIQUE(epic, snapshot_time_utc)` per table
  (resolution no longer needs to be part of the key — it's fixed per table).
- **Write mode:** `INSERT OR IGNORE`. Existing rows are **never updated** —
  re-downloading a still-forming candle keeps the stale row already in
  the DB.

In [5]:
# --- SQLite (flattened prices, same schema as 03_view_ig_prices.ipynb) --------------------
def write_db():
    if not cfg.save_db:
        print("SAVE_DB is False - skipping SQLite upsert")
        return

    from igmarket.candle_csv import candle_to_row
    from igmarket.candle_db import INSERT_COLUMNS, init_candles_table
    from igmarket.constants.ig_candle_fields import CandleField

    conn = sqlite3.connect(cfg.db_path)
    table = init_candles_table(conn, cfg.resolution)
    rows = [
        (cfg.epic, *candle_to_row(c))
        for c in raw_candles
        if c.get(CandleField.SNAPSHOT_TIME_UTC)
    ]
    before = conn.total_changes
    placeholders = ", ".join("?" for _ in INSERT_COLUMNS)
    conn.executemany(
        f"INSERT OR IGNORE INTO {table} ({', '.join(INSERT_COLUMNS)}) "
        f"VALUES ({placeholders})",
        rows,
    )
    conn.commit()
    inserted = conn.total_changes - before
    print(f"{cfg.db_path.resolve()}: +{inserted} new row(s) in `{table}` "
          f"({len(rows) - inserted} already present)")
    conn.close()


write_db()

SAVE_DB is False - skipping SQLite upsert


## 4. preview + close (mid) chart

Needs `pandas` and `matplotlib` (`%pip install pandas matplotlib`). Builds a
frame straight from the candles just downloaded (not from disk), so it works
whether or not `SAVE_CSV` / `SAVE_DB` wrote anything this run: a tail-10 table
+ `describe()`, then a close (mid) line chart with a shaded high/low band.

In [ ]:
try:
    import pandas as pd
except ModuleNotFoundError:
    print("pandas not installed - run:  %pip install pandas")
    df = None
else:
    from igmarket.candle_csv import candle_to_row
    from igmarket.constants.csv_headers import CSV_HEADERS

    # Straight from the candles just downloaded, so the preview and chart work
    # even when SAVE_CSV and SAVE_DB are both False (nothing written to disk).
    df = pd.DataFrame((candle_to_row(c) for c in raw_candles), columns=CSV_HEADERS)
    df["snapshot_time_utc"] = pd.to_datetime(df["snapshot_time_utc"])
    df = df.set_index("snapshot_time_utc").sort_index()
    if df.empty:
        print(f"Download returned 0 {cfg.resolution} candles - nothing to preview.")
    else:
        display(df.tail(10))
        display(df.describe())

In [ ]:
if df is None:
    print("Run the pandas cell above first.")
elif df.empty:
    print(
        f"No {cfg.resolution} candles for the last {cfg.days_back} day(s) - "
        f"nothing to plot (e.g. a weekend / market-closed window); "
        f"try a larger DAYS_BACK."
    )
else:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        print("matplotlib not installed - run:  %pip install matplotlib")
    else:
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(df.index, df["close_mid_price"], color="#26a69a")
        ax.fill_between(df.index, df["low_mid_price"], df["high_mid_price"], color="#26a69a", alpha=0.15)
        ax.set_title(f"{cfg.epic}  {cfg.resolution}  close (mid)  -  last {cfg.days_back} day(s)")
        ax.set_ylabel("price (mid)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()